In [0]:
%run ../00_common/data_utils

In [0]:
batch_id =  dbutils.widgets.get("batch_id")
print(f"batch_id: {batch_id}")

blob_storage_name = get_env_config("sap_touchpoint_blob_config.blob_storage_name")
blob_storage_key = get_env_config("sap_touchpoint_blob_config.blob_storage_key")
blob_container_name = get_env_config("sap_touchpoint_blob_config.blob_container_name")

print(f"blob_storage_name：{blob_storage_name}")
print(f"blob_container_name：{blob_container_name}")

In [0]:
def list_source_files(source_path, archive_path, wasbs_source_path, wasbs_archive_path):
    """
    筛选Blob中TouchPointList_开头、.csv结尾的文件，返回按修改时间降序的文件列表
    """
    try:
        print(f"\n[{get_now_cst('%Y-%m-%d %H:%M:%S')}] [STEP1] 开始筛选Blob文件...")

        # 用Spark SQL LIST获取文件列表
        file_df = spark.sql(f"LIST '{wasbs_source_path}'")
        print(f"[{get_now_cst('%Y-%m-%d %H:%M:%S')}] [STEP1] LIST命令返回列名：{file_df.columns}")

        # 筛选条件：文件（非目录）+ TouchPointList_开头 + .csv结尾
        filtered_df = file_df.filter(
            (~F.col("path").endswith("/")) &
            (F.col("path").like(f"%{source_path}TouchPointList_%")) &
            (F.col("path").endswith(".csv"))
        )

        # 按修改时间降序排序
        file_list = filtered_df.orderBy(F.col("modification_time").desc()).collect()
        print(f"[{get_now_cst('%Y-%m-%d %H:%M:%S')}] [STEP1] 源目录总文件数：{file_df.count()} | 符合条件文件数：{len(file_list)}")

        # 封装文件信息+归档路径映射
        result_list = []
        path_mapping = {}
        for row in file_list:
            file_full_path = row["path"]
            file_name = file_full_path.split(f"{blob_container_name}@{blob_storage_name}.blob.core.windows.net/")[1]
            file_last_modified = datetime.fromtimestamp(row["modification_time"] / 1000)

            # 封装文件信息
            file_info = {
                "name": file_name,
                "full_path": file_full_path,
                "last_modified": file_last_modified
            }
            result_list.append(file_info)

            # 构建归档路径
            archive_file_name = file_name.replace(source_path, archive_path)
            archive_full_path = f"{wasbs_archive_path}{archive_file_name.split('/')[-1]}"
            path_mapping[file_full_path] = archive_full_path

            print(f"[{get_now_cst('%Y-%m-%d %H:%M:%S')}] [STEP1] 筛选出文件：{file_name} | 修改时间：{file_last_modified.strftime('%Y-%m-%d %H:%M:%S')}")

        return result_list, path_mapping

    except Exception as e:
        print(f"[{get_now_cst('%Y-%m-%d %H:%M:%S')}] [ERROR] 筛选文件失败：{str(e)}")
        traceback.print_exc()
        return [], {}

In [0]:
def load_data_to_table(sap_file_path, where_str, flag):
    source_type = flag.upper()  # "APAC" 或 "KOR"
    sap_file_name = sap_file_path.split("/")[-1]
    source_df = None

    try:
        source_df = load_csv_data_from_blob(blob_storage_name, blob_storage_key, sap_file_path)
        print(f"source_df count: {source_df.count()}")

    except AnalysisException as e:
            if "Path does not exist" in str(e):
                traceback.print_exc()
                print(f"=>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>> \n{sap_file_name}文件未找到 ")
                source_df = None
            else:
                raise
    except Exception as e:
        raise
    
    # 从文件名中提取日期，兼容：
    # TouchPointList_SAPBI_2026-04-11.csv
    # TouchPointList_SAPBI_KOR_2026-04-11.csv
    date_match = re.search(r"_(\d{4}-\d{2}-\d{2}).*\.csv$", sap_file_name)
    file_date = date_match.group(1) if date_match else None
    if file_date is None:
        print(f"[WARN] 无法从文件名提取日期：{sap_file_name}")

    if source_df != None:
        sap_df = (source_df
            .select(
                F.expr("uuid()").alias("TP_SAPBI_ID"),
                F.col("Sales Organization - Market (Text)").alias("MarketCode"),
                F.col("Sales Organization").alias("SalesOrganisation"),
                F.col("Division - Key").alias("DivisionCode"),
                F.col("Customer number - Key").alias("CustomerNumber"),
                F.col("Customer number - Long Text").alias("CustomerName"),
                F.col("Business Type - Text").alias("BusinessType"),
                F.col("Customer number - Attribute 2 - Custom (Key)").alias("RetailerOnlineDoor"),
                F.col("Customer group - Text").alias("CustomerGroup"),
                F.col("Global Reporting Group - Key").alias("ReportingGroup_Global_Code"),
                F.col("Global Reporting Group - Medium Text").alias("ReportingGroup_Global_Name"),
                F.col("Regional Reporting Group - Key").alias("ReportingGroup_Regional_Code"),
                F.col("Regional Reporting Group - Long Text").alias("ReportingGroup_Regional_Name"),
                F.col('Affiliate/Market Reporting Group, - Key').alias("ReportingGroup_Affiliate_Code"),
                F.col('Affiliate/Market Reporting Group, - Medium Text').alias("ReportingGroup_Affiliate_Name"),
                F.col("Retailer").alias("Retailer"),
                F.col("Customer number - Region (Text)").alias("Region"),
                F.col("Customer number - City (Key)").alias("City"),
                F.lit(sap_file_name).alias("Filename_SAPBI"),
                F.current_timestamp().alias("CREATION_DT"),
                F.current_timestamp().alias("UPDATE_DT"),
                F.lit(batch_id).alias("BATCH_ID"),
                F.lit(source_type).alias("source_type"),
                F.lit(file_date).alias("file_date")
            )
        )

        source_table = f"{get_env_config('silver_touchpoint_parsed_database')}.t_touchpoint_sapbi_source"

        # 先落地原始数据到source表
        # sap_df.write.option("mergeSchema", "true").mode("append").saveAsTable(source_table)
        append_table(sap_df, source_table)
        

In [0]:
def move_files_to_archive(path_mapping):
    """将文件移动到归档目录"""
    try:
        print(f"\n[{get_now_cst('%Y-%m-%d %H:%M:%S')}] [STEP3] 开始归档文件...")
        print(f"[{get_now_cst('%Y-%m-%d %H:%M:%S')}] [STEP3] 待归档文件数：{len(path_mapping)}")

        success_count = 0
        fail_count = 0
        fail_files = []

        for source_path, archive_path in path_mapping.items():
            try:
                # 校验路径
                if not source_path or not archive_path:
                    fail_count += 1
                    fail_files.append(source_path)
                    continue

                # 优化文件存在性判断（避免ls报错）
                try:
                    if dbutils.fs.ls(archive_path):
                        print(f"[{get_now_cst('%Y-%m-%d %H:%M:%S')}] [STEP3] 归档路径已存在，删除后再移动：{archive_path}")
                        dbutils.fs.rm(archive_path, recurse=True)
                except Exception as e:
                    if "java.io.FileNotFoundException" not in str(e):
                        print(f"[{get_now_cst('%Y-%m-%d %H:%M:%S')}] [WARN] 检查归档路径失败：{archive_path} | 错误：{str(e)}")

                # 执行移动
                dbutils.fs.mv(source_path, archive_path)
                success_count += 1
                print(f"[{get_now_cst('%Y-%m-%d %H:%M:%S')}] [STEP3] 成功归档：{source_path} → {archive_path}")
            except Exception as e:
                fail_count += 1
                fail_files.append(source_path)
                print(f"[{get_now_cst('%Y-%m-%d %H:%M:%S')}] [STEP3] 归档失败：{source_path} | 错误：{str(e)}")

        print(f"\n[{get_now_cst('%Y-%m-%d %H:%M:%S')}] [STEP3] 归档完成！成功：{success_count} | 失败：{fail_count}")
        if fail_files:
            print(f"[{get_now_cst('%Y-%m-%d %H:%M:%S')}] [STEP3] 失败文件列表：{fail_files}")
        return success_count == len(path_mapping)

    except Exception as e:
        print(f"[{get_now_cst('%Y-%m-%d %H:%M:%S')}] [ERROR] 归档流程失败：{str(e)}")
        traceback.print_exc()
        return False

In [0]:
def ingest_t_touchpoint_sapbi(source_path, archive_path, flag):
    wasbs_source_path = f"wasbs://{blob_container_name}@{blob_storage_name}.blob.core.windows.net/{source_path}"
    wasbs_archive_path = f"wasbs://{blob_container_name}@{blob_storage_name}.blob.core.windows.net/{archive_path}"
    print(f"source_path: {source_path}")
    print(f"archive_path: {archive_path}")
    print(f"wasbs_source_path：{wasbs_source_path}")
    print(f"wasbs_archive_path：{wasbs_archive_path}")
    
    # ========== 步骤1：配置Spark连接Blob ==========
    print(f"[{get_now_cst('%Y-%m-%d %H:%M:%S')}] [INIT] 配置Spark连接Azure Blob...")
    spark.conf.set(f"fs.azure.account.key.{blob_storage_name}.blob.core.windows.net", blob_storage_key)
    print(f"[{get_now_cst('%Y-%m-%d %H:%M:%S')}] [INIT] Spark Blob连接配置完成！")
    
    
    # ========== 步骤2：筛选最新文件 ==========
    file_list, path_mapping = list_source_files(source_path, archive_path, wasbs_source_path, wasbs_archive_path)
    # 无符合条件文件，终止任务
    if not file_list:
        print(f"[{get_now_cst('%Y-%m-%d %H:%M:%S')}] [WARNING] 未找到任何TouchPointList_开头 .csv结尾的文件，任务终止")
        return {}
    
    latest_file = file_list[0]
    latest_file_path = latest_file["full_path"]
    latest_file_name = latest_file["name"].split("/")[-1]
    print(f"\n[{get_now_cst('%Y-%m-%d %H:%M:%S')}] [STEP2] 选中最新文件：{latest_file_name} | 路径：{latest_file_path}")
    
    
    # ========== 步骤3：最新文件写入databricks ==========
    if flag == "apac":
        load_data_to_table(latest_file_path, "MarketCode != 'Korea' ", flag)
    else:
        load_data_to_table(latest_file_path, "MarketCode = 'Korea' ", flag)

    # 归档放在脚本最后统一处理
    return path_mapping

In [0]:
def main():
    # append to t_touchpoint_sapbi_source - apac
    print("============================== apac ==============================")
    apac_source_path = get_env_config("sap_touchpoint_blob_config.apac_sap_file_path")
    apac_archive_path = get_env_config("sap_touchpoint_blob_config.apac_sap_file_archive_path")
    apac_path_mapping = ingest_t_touchpoint_sapbi(apac_source_path, apac_archive_path, "apac")

    # append to t_touchpoint_sapbi_source - kor
    print("============================== kor ==============================")
    kor_source_path = get_env_config("sap_touchpoint_blob_config.kor_sap_file_path")
    kor_archive_path = get_env_config("sap_touchpoint_blob_config.kor_sap_file_archive_path")
    kor_path_mapping = ingest_t_touchpoint_sapbi(kor_source_path, kor_archive_path, "kor")

    # archive files
    all_path_mapping = {}
    all_path_mapping.update(apac_path_mapping or {})
    all_path_mapping.update(kor_path_mapping or {})

    if all_path_mapping:
        move_files_to_archive(all_path_mapping)
    else:
        print(f"[{get_now_cst('%Y-%m-%d %H:%M:%S')}] [INFO] 无需归档文件")

In [0]:
with StepLogger("ingest_blob_t_touchpoint_sapbi", "01-2", "touchpoint", task_id=batch_id) as logger:
    main()